# 🔌 API Exploration Lab
Day 2 – Session 2

In this notebook we will authenticate, send API requests, inspect responses, and implement simple robustness patterns.

## 1) Setup & Credentials

In [ ]:
import os, json
from typing import Dict, Any
import time, random

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


✅ OPENAI_API_KEY is set.
✅ GOOGLE_API_KEY is set.


## 2) Helper: Safe API Generate Function with Retries

In [2]:
import abc

class GenerativeAIClient(abc.ABC):

  @abc.abstractmethod
  def generate(self, prompt: str, **kwargs) -> str:
      ...

In [ ]:
from dataclasses import dataclass
from openai import OpenAI

@dataclass
class OpenAIClient(GenerativeAIClient):
  model: str='gpt-4o-mini'
  temperature: float=0.7
  max_tokens: int=200
  retries: int=3
  backoff: float=0.8

  def __post_init__(self):
      OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
      if not OPENAI_API_KEY:
          raise ValueError("OPENAI_API_KEY is not set in the environment")

      self.client = OpenAI(api_key=OPENAI_API_KEY)

  def generate(self, prompt: str) -> str:
      """Call the chat completion API with basic retries and timing.
      Returns the model's answer as plain text.
      """
      response = self.client.chat.completions.create(
          model=self.model,
          messages=[
              {"role": "user", "content": prompt}
          ],
          temperature=self.temperature,
          max_tokens=self.max_tokens
      )
      if response is None:
          raise ValueError("No response from the API")

      choices = response.choices
      if not choices or len(choices) == 0:
          raise ValueError("Failed to get a valid response from the API")

      first_choice = choices[0]
      message = first_choice.message
      if message.role != "assistant":
          raise ValueError("Invalid message format in the response")

      if not message.content:
          reason = message.refusal
          raise ValueError("No content in the assistant's message: " + str(reason))
      
      return message.content

    

In [ ]:
from google import genai
from google.genai.types import GenerateContentConfig
from dataclasses import dataclass

@dataclass
class GoogleGenAIClient(GenerativeAIClient):
  model: str='gemini-2.5-flash'
  temperature: float=0.7
  max_tokens: int=200
  retries: int=3
  backoff: float=0.8

  def __post_init__(self):
      GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
      if not GOOGLE_API_KEY:
          raise ValueError("GOOGLE_API_KEY is not set in the environment")
      self.client = genai.Client(api_key=GOOGLE_API_KEY)
      self.config = GenerateContentConfig(temperature=self.temperature, max_output_tokens=self.max_tokens)

  def generate(self, prompt: str) -> str:
              
      """Call the chat completion API with basic retries and timing.
      Returns the model's answer as plain text.
      """

      if not isinstance(prompt, str):
          raise ValueError("Prompt should be a string")

      response = self.client.models.generate_content(model=self.model, contents=prompt, config=self.config)
      if response is None:
          raise ValueError("No response from the API")
      
      if not response.text:
          if response.candidates:
              reason = response.candidates[0].finish_reason
              if reason == "MAX_TOKENS":
                  raise ValueError("Response was cut off due to max tokens limit.")
              raise ValueError(f"No content in the response: {reason}")
          raise ValueError("Failed to get a valid response from the API")
      
      return response.text



In [6]:
client = OpenAIClient()
response = client.generate("Hello, how are you?")
print(response)

Hello! I'm just a program, but I'm here and ready to help you. How can I assist you today?


In [ ]:
client = GoogleGenAIClient()
response = client.generate("Hello, how are you?", max_tokens=500)
print(response)

In [ ]:
client = GenerativeAIClient()


In [ ]:
from enum import Enum

class Provider(Enum):
    OPENAI = 'openai'
    GOOGLE = 'google'

class GenerativeAIClientFactory:
    
    @staticmethod
    def get_client(provider: Provider) -> GenerativeAIClient:
        if provider == Provider.OPENAI:
            return OpenAIClient()
        elif provider == Provider.GOOGLE:
            return GoogleGenAIClient()
        else:
            raise ValueError(f"Unknown provider: {provider}")

# static vs class methods

In [45]:
class A:

  def __init__(self):
      self.value = 0

  def increment(self):
      self.value += 1

  def show(self):
      print(self.value)

  def __str__(self):
      return f"str A(value={self.value})"
  
  def __repr__(self):
      return f"repr A(value={self.value})"


In [46]:
a = A()
a


repr A(value=0)

In [71]:
from dataclasses import dataclass

@dataclass
class B:
  value: int
  count: int
  name: str
  parts: list

In [1]:
from pydantic import BaseModel

class B(BaseModel):
  value: int
  count: int
  name: str
  parts: list

DTO: Data Transfer Object

In [2]:
b = B(value=0, count=0, name='default', parts=["part1", "part2"])
b

B(value=0, count=0, name='default', parts=['part1', 'part2'])

In [6]:
import json
print(json.dumps(b.model_json_schema(), indent=2))

{
  "properties": {
    "value": {
      "title": "Value",
      "type": "integer"
    },
    "count": {
      "title": "Count",
      "type": "integer"
    },
    "name": {
      "title": "Name",
      "type": "string"
    },
    "parts": {
      "items": {},
      "title": "Parts",
      "type": "array"
    }
  },
  "required": [
    "value",
    "count",
    "name",
    "parts"
  ],
  "title": "B",
  "type": "object"
}


In [82]:
s2 = b.model_dump_json()
s2

'{"value":0,"count":0,"name":"default","parts":["part1","part2"]}'

In [96]:
s2 = '{"value":0,"count": "zero", "name":"default","parts":["part1","part2"]}'
json.loads(s2)

{'value': 0, 'count': 'zero', 'name': 'default', 'parts': ['part1', 'part2']}

In [97]:
c2 = B.model_validate_json(s2)
c2

ValidationError: 1 validation error for B
count
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='zero', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [86]:
c2.parts

['part1', 'part2']

In [79]:
d = b.__dict__
d

{'value': 0, 'count': 0, 'name': 'default', 'parts': ['part1', 'part2']}

In [74]:
import json

s = json.dumps(d)
s

'{"value": 0, "count": 0, "name": "default", "parts": ["part1", "part2"]}'

In [ ]:
d2 = json.loads(s)
d2

{'value': 0, 'count': 'zero', 'name': 'default', 'parts': ['part1', 'part2']}

In [ ]:
c = B(**d2)
c

ValidationError: 1 validation error for B
count
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='zero', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [70]:
c.name

'default'

## 3) Compare Parameters

In [ ]:
prompt = 'Write three product taglines for a note-taking app.'

try:
    out1 = generate(prompt, temperature=0.2)
    out2 = generate(prompt, temperature=0.9)
    print('— Low temperature (0.2):\n', out1)
    print('\n— High temperature (0.9):\n', out2)

except Exception as e:
    raise # one would take the opportunity to do something more meaningful...

## 4) Add usage and latency metadata

Modify the function above so that it returns usage and latency.

In [ ]:
def generate(prompt: str, *, model: str='gpt-4o-mini', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> Dict[str, Any]:
    """Call the chat completion API with basic retries and timing.
    Returns dict with text, usage (if available), and latency_ms.
    """
    ...

out = generate(prompt)
print('Usage:', json.dumps(out['usage']))
print('Latency (ms):', out['latency_ms'])

## 5) More exercises

1. Modify `generate` to accept a list of messages (system, user) and a stop sequence.
2. Add structured logging (JSON lines).
3. Try two prompts and compare responses for tone and length.